# SARM Sanity Test (200 steps)

**目的**：在跑 5000 步全量訓練之前，先用 200 步快速驗證修好的 dataset 確實會讓 dense head 學到東西。

**為什麼需要這個 notebook**：
- 原本的 `Lebruhbruh/SARM-opendata-annotated-fixed` 因為 `meta/episodes/*.parquet` 只有 8/273 個 episode 有 dense 標注，訓出來的 model 預測全 0
- 已經跑 `materialize_dense_annotations.py` 把 `lerobot_annotations.json` 內容寫進 parquet 並重 push 上 Hub（v3.0 tag 已移動）
- 這個 notebook：(a) 從 Hub assert 273/273 都有標注、(b) 訓 200 步、(c) 推到獨立的 `-sanity` repo、(d) inference 確認預測非全 0

**過關條件**：
1. Cell 6 印出 `dense_subtask_names empty: 0/273`
2. Cell 10 的訓練 log：`dense_stage_loss` 跟 `dense_subtask_loss` 從第一個 log step 就 > 0 且持續下降
3. Cell 11 印出 ep 21 中段 frame 的 `dense reward` ∈ [0.1, 0.9]（不是 0 也不是 1）

若上述都過，再回 `SARM_Training_Colab.ipynb` 跑 5000 步正式訓練。

In [ ]:
# Cell 0: GPU 確認
import subprocess, torch
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
assert torch.cuda.is_available(), '錯誤：沒有 GPU！請檢查 Runtime 設定'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')
if vram_gb < 16:
    print('WARNING: VRAM < 16GB，200 steps sanity 應該還行但會慢')
else:
    print('✓ GPU 規格符合需求')

In [ ]:
# Cell 1: 使用者設定（必填）
# ============================================================
HF_USERNAME = 'Lebruhbruh'
HF_TOKEN    = ''   # <-- 貼上你的 Write Token

# 修好的 dataset（已 push 過、v3.0 tag 已移動到正確 commit）
ANNOTATED_DATASET = 'Lebruhbruh/SARM-opendata-annotated-fixed'

# Sanity 訓練 push 到獨立 repo，不污染 production 的 sarm-charging-bimanual
MODEL_REPO_ID = f'{HF_USERNAME}/sarm-charging-bimanual-sanity'

VIDEO_KEY = 'observation.images.top'

SUBTASK_NAMES = [
    'Pick up the phone',
    'Flip the phone sideways',
    'Pick up the charging cable and plug it into the phone',
    'Turn on the power of the extension cord',
]

# Sanity 訓練參數（短）
SANITY_STEPS = 200
SANITY_SAVE_FREQ = 200

# Sanity inference 用的 episodes（原本完全沒標注的，最強驗證）
EVAL_EPISODE = 21   # 用 ep 21（你原本就想 viz 的）跑單一 frame inference

assert HF_TOKEN, '請填 HF_TOKEN'
print(f'標注資料集: {ANNOTATED_DATASET}')
print(f'Sanity model: {MODEL_REPO_ID}')
print(f'相機:       {VIDEO_KEY}')
print(f'步數:       {SANITY_STEPS} steps')
print(f'Inference 驗證 ep: {EVAL_EPISODE}')

In [ ]:
# Cell 1b: 進度回饋工具
import threading, time, contextlib
from datetime import timedelta

def _fmt_elapsed(seconds: float) -> str:
    return str(timedelta(seconds=int(seconds)))

@contextlib.contextmanager
def progress_stage(title: str, steps=None, heartbeat_seconds: int = 30):
    print(f'\n━━━ {title} ━━━')
    if steps:
        for i, step in enumerate(steps, 1):
            print(f'  {i}) {step}')
        print()
    stop_evt = threading.Event()
    start = time.monotonic()
    def _heartbeat():
        while not stop_evt.wait(heartbeat_seconds):
            print(f'  ⏱  已執行 {_fmt_elapsed(time.monotonic() - start)}', flush=True)
    th = threading.Thread(target=_heartbeat, daemon=True)
    th.start()
    error = None
    try:
        yield
    except BaseException as e:
        error = e
        raise
    finally:
        stop_evt.set()
        th.join(timeout=1)
        total = _fmt_elapsed(time.monotonic() - start)
        if error is None:
            print(f'\n✓ {title} 完成，總耗時 {total}')
        else:
            print(f'\n✗ {title} 失敗（{type(error).__name__}），已執行 {total}')

print('✓ progress_stage 已就緒')

In [ ]:
# Cell 2: 安裝 LeRobot + SARM
import subprocess, sys

with progress_stage(
    '安裝 LeRobot + SARM（預計 2-5 分鐘）',
    steps=['pip install lerobot[sarm] from GitHub', '確認版本'],
    heartbeat_seconds=30,
):
    r = subprocess.run(
        'pip install -q "git+https://github.com/huggingface/lerobot.git#egg=lerobot[sarm]"',
        shell=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f'pip install 失敗 ({r.returncode})')
    print('  ✓ LeRobot 安裝完成')
    check_cmd = 'import lerobot, transformers; print(lerobot.__version__); print(transformers.__version__)'
    r = subprocess.run([sys.executable, '-c', check_cmd], capture_output=True, text=True)
    versions = r.stdout.strip().split()
    print(f'  LeRobot: {versions[0]}')
    print(f'  Transformers: {versions[1] if len(versions)>1 else "?"}')

In [ ]:
# Cell 3: Transformers 5.x patch + HF login
import os, glob, torch, subprocess
import lerobot
from huggingface_hub import login

lerobot_root = os.path.dirname(lerobot.__file__)
hits = glob.glob(os.path.join(lerobot_root, '**', 'processor_sarm.py'), recursive=True)
if not hits:
    hits = glob.glob(os.path.join(os.path.dirname(lerobot_root), '**', 'processor_sarm.py'), recursive=True)
if not hits:
    raise FileNotFoundError('找不到 processor_sarm.py')
processor_path = hits[0]
with open(processor_path) as f:
    content = f.read()
patched = False
indent = '\n        '
old_img = 'embeddings = self.clip_model.get_image_features(**inputs).detach().cpu()'
new_img = indent.join([
    'output = self.clip_model.get_image_features(**inputs)',
    'if not isinstance(output, torch.Tensor):',
    '    output = output.pooler_output',
    'embeddings = output.detach().cpu()',
])
if old_img in content:
    content = content.replace(old_img, new_img)
    patched = True
old_txt = 'embeddings = self.clip_model.get_text_features(**inputs).detach().cpu()'
new_txt = indent.join([
    'output = self.clip_model.get_text_features(**inputs)',
    'if not isinstance(output, torch.Tensor):',
    '    output = output.pooler_output',
    'embeddings = output.detach().cpu()',
])
if old_txt in content:
    content = content.replace(old_txt, new_txt)
    patched = True
if patched:
    with open(processor_path, 'w') as f:
        f.write(content)
    print('✓ processor_sarm.py patched')
else:
    print('✓ processor_sarm.py 已是新版，無需 patch')

login(token=HF_TOKEN)
print('✓ HuggingFace 登入成功')

In [ ]:
# Cell 4: 清空 Colab 端的 dataset cache（關鍵！）
# ============================================================
# 如果 Colab 之前跑過訓練，本地 cache 會是 push 前的舊版 parquet。
# 不清掉的話 LeRobotDataset 直接讀 cache，會錯過剛 push 上去的修正。
# 清掉之後 Cell 5 的 download 會強制重抓 v3.0。
# ============================================================
import shutil, os

cache_paths = [
    os.path.expanduser('~/.cache/huggingface/lerobot/hub/datasets--Lebruhbruh--SARM-opendata-annotated-fixed'),
    os.path.expanduser('~/.cache/huggingface/hub/datasets--Lebruhbruh--SARM-opendata-annotated-fixed'),
]
for p in cache_paths:
    if os.path.isdir(p):
        shutil.rmtree(p)
        print(f'✓ 清掉 cache: {p}')
    else:
        print(f'  (不存在，跳過): {p}')
print('\n→ 下一個 cell 會強制重抓 v3.0 parquet')

In [ ]:
# Cell 5: 強制從 Hub 重抓 + assert dense annotations 完整
# ============================================================
# 在訓練前就驗證 v3.0 tag 指向的 parquet 是修好的版本，避免訓練跑一半才發現。
# 用 force_download=True 確保不會吃 cache。
# ============================================================
import pandas as pd
from huggingface_hub import hf_hub_download

p = hf_hub_download(
    ANNOTATED_DATASET,
    'meta/episodes/chunk-000/file-000.parquet',
    repo_type='dataset',
    revision='v3.0',
    force_download=True,
    token=HF_TOKEN,
)
df = pd.read_parquet(p)

def count_empty(col):
    return sum(1 for v in df[col]
               if v is None or (hasattr(v, '__len__') and len(v) == 0))

empty_dense  = count_empty('dense_subtask_names')
empty_sparse = count_empty('sparse_subtask_names')
empty_plain  = count_empty('subtask_names')

print(f'rows: {len(df)}')
print(f'dense_subtask_names  empty: {empty_dense}/{len(df)}')
print(f'sparse_subtask_names empty: {empty_sparse}/{len(df)}')
print(f'subtask_names        empty: {empty_plain}/{len(df)}')

assert empty_dense == 0, f'⚠ Hub v3.0 還是有 {empty_dense} 個 dense_subtask_names 空的；重跑 materialize_dense_annotations.py'
assert empty_sparse == 0, f'⚠ sparse_subtask_names {empty_sparse} 空'
assert empty_plain == 0,  f'⚠ subtask_names {empty_plain} 空'

# Spot check 中段一筆
row = df.iloc[150]
end_f = list(row['dense_subtask_end_frames'])
length = int(row['length'])
print(f'\nspot check ep 150: length={length}, dense_end_frames={end_f}, coverage={end_f[-1]/length:.3f}')
assert end_f[-1] / length >= 0.95, f'ep 150 覆蓋率 < 95%'

print('\n✓ Hub v3.0 parquet 完整，可以訓練')

In [ ]:
# Cell 6: AV1 codec patch（避免 cv2 在 Colab 讀不到影片）
import os, glob, cv2

# 先確認 LeRobotDataset 已經能 lazy-load（觸發 metadata 抓影片路徑）
from lerobot.datasets.lerobot_dataset import LeRobotDataset
_ = LeRobotDataset(ANNOTATED_DATASET)

test_paths = (
    sorted(glob.glob(
        '/root/.cache/huggingface/lerobot/hub/'
        'datasets--Lebruhbruh--SARM-opendata-annotated-fixed/snapshots/*/'
        'videos/observation.images.top/chunk-000/*.mp4'
    ))
    or sorted(glob.glob(
        '/root/.cache/huggingface/hub/'
        'datasets--Lebruhbruh--SARM-opendata-annotated-fixed/snapshots/*/'
        'videos/observation.images.top/chunk-000/*.mp4'
    ))
)
cv2_can_read_av1 = False
if test_paths:
    cap = cv2.VideoCapture(test_paths[0])
    if cap.isOpened():
        ret, frame = cap.read()
        cv2_can_read_av1 = bool(ret and frame is not None)
    cap.release()
    print(f'AV1 test ({os.path.basename(test_paths[0])}): cv2 {"OK" if cv2_can_read_av1 else "FAIL"}')
else:
    print('⚠ 找不到已下載的 mp4，可能 LeRobotDataset 還沒 lazy-fetch；訓練時會自動補')

if not cv2_can_read_av1:
    print('  → 訓練底層用 torchcodec 解碼，跟 cv2 無關，可以繼續')
else:
    print('✓ cv2 也讀得到')

In [ ]:
# Cell 7: Sanity 訓練（200 steps，預計 5-10 分鐘）
# ============================================================
# 跟正式訓練一模一樣的參數，只差 --steps=200 跟 --reward_model.repo_id 是 -sanity
# ============================================================
import subprocess, os, sys, shutil

env = os.environ.copy()
env.update({
    'HF_TOKEN': HF_TOKEN,
    'HUGGING_FACE_HUB_TOKEN': HF_TOKEN,
    'PYTHONUNBUFFERED': '1',
})

lerobot_bin = shutil.which('lerobot-train')
lerobot_train = [lerobot_bin] if lerobot_bin else [sys.executable, '-m', 'lerobot.scripts.lerobot_train']

out_dir = 'outputs/train/sarm_charging_bimanual_sanity'
if os.path.isdir(out_dir):
    shutil.rmtree(out_dir, ignore_errors=True)

cmd = [
    *lerobot_train,
    f'--dataset.repo_id={ANNOTATED_DATASET}',
    '--reward_model.type=sarm',
    '--reward_model.annotation_mode=dense_only',
    f'--reward_model.image_key={VIDEO_KEY}',
    '--reward_model.state_key=observation.state',
    '--reward_model.n_obs_steps=8',
    '--reward_model.frame_gap=30',
    f'--reward_model.repo_id={MODEL_REPO_ID}',
    '--reward_model.push_to_hub=true',
    f'--output_dir={out_dir}',
    '--batch_size=64',
    f'--steps={SANITY_STEPS}',
    f'--save_freq={SANITY_SAVE_FREQ}',
    '--tolerance_s=0.001',
    '--wandb.enable=false',
]
print('cmd:', ' '.join(cmd))

log_path = '/content/sarm_sanity_train.log'
loss_history = []   # 蒐集 dense_* loss 給 Cell 8 判讀

with progress_stage(
    f'Sanity 訓練 {SANITY_STEPS} steps（預計 5-10 分鐘）',
    steps=[
        f'載入 {ANNOTATED_DATASET} v3.0',
        f'訓練 {SANITY_STEPS} steps（每步 print 一次 loss）',
        f'推到 {MODEL_REPO_ID}',
    ],
    heartbeat_seconds=30,
):
    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    with open(log_path, 'w') as logf:
        for line in proc.stdout:
            print(line, end='', flush=True)
            logf.write(line)
            logf.flush()
            # 順手抓 loss 行
            if 'dense_stage_loss' in line or 'dense_subtask_loss' in line:
                loss_history.append(line.strip())
    returncode = proc.wait()

assert returncode == 0, f'訓練失敗 (returncode={returncode})，看 {log_path}'
print(f'\n✓ 訓練完成，model 已推到 https://huggingface.co/{MODEL_REPO_ID}')

In [ ]:
# Cell 8: 解析訓練 log 確認 dense_* loss 有在掉
# ============================================================
# 從 /content/sarm_sanity_train.log 抓所有 step 的 dense_stage_loss / dense_subtask_loss，
# 看 first/last 對比。若 last < first * 0.7 就算「有學東西」。
# ============================================================
import re, json

log_lines = open('/content/sarm_sanity_train.log').read().splitlines()

# lerobot-train 的 log 格式不固定（json or k=v）；用 regex 抓 step 與兩個 dense loss
step_pat = re.compile(r'\bstep[:=]\s*(\d+)', re.I)
dstage_pat = re.compile(r'dense_stage_loss[:=]?\s*([0-9.eE+-]+)')
dsubtask_pat = re.compile(r'dense_subtask_loss[:=]?\s*([0-9.eE+-]+)')

samples = []  # (step, dstage, dsubtask)
for ln in log_lines:
    s = step_pat.search(ln)
    a = dstage_pat.search(ln)
    b = dsubtask_pat.search(ln)
    if s and (a or b):
        samples.append((
            int(s.group(1)),
            float(a.group(1)) if a else None,
            float(b.group(1)) if b else None,
        ))

if not samples:
    print('⚠ 沒抓到 dense loss 的 log；可能 lerobot-train 的 log 格式變了')
    print('  手動翻一下 /content/sarm_sanity_train.log 看 dense_stage_loss / dense_subtask_loss')
else:
    print(f'共抓到 {len(samples)} 筆 loss log\n')
    print(f"{'step':>6}  {'dense_stage_loss':>18}  {'dense_subtask_loss':>20}")
    print('-' * 50)
    for s, a, b in samples[:5] + (samples[-5:] if len(samples) > 10 else []):
        print(f'{s:>6}  {a if a is not None else "-":>18}  {b if b is not None else "-":>20}')

    first_dstage = next((a for _, a, _ in samples if a is not None), None)
    last_dstage  = next((a for _, a, _ in reversed(samples) if a is not None), None)
    first_dsub   = next((b for _, _, b in samples if b is not None), None)
    last_dsub    = next((b for _, _, b in reversed(samples) if b is not None), None)

    print(f'\ndense_stage_loss:   {first_dstage} → {last_dstage}')
    print(f'dense_subtask_loss: {first_dsub} → {last_dsub}')

    issues = []
    if first_dstage is not None and first_dstage < 1e-4:
        issues.append('first dense_stage_loss 接近 0 → dataset 標注沒生效，model 仍學「全 0」')
    if last_dstage is not None and first_dstage is not None and last_dstage > first_dstage * 0.9:
        issues.append(f'dense_stage_loss 只掉 {(1-last_dstage/first_dstage)*100:.1f}%（< 10%）→ 學得太少')
    if last_dsub is not None and first_dsub is not None and last_dsub > first_dsub * 0.9:
        issues.append(f'dense_subtask_loss 只掉 {(1-last_dsub/first_dsub)*100:.1f}% → 學得太少')

    if issues:
        print('\n⚠ 警告：')
        for i in issues:
            print(f'   - {i}')
    else:
        print('\n✓ Loss 走勢正常（兩個 dense loss 都有持續下降）')

In [ ]:
# Cell 9: Inference sanity — 對 ep 21 中段 frame 跑一次預測，確認 reward 不是 0
# ============================================================
# 載剛訓好的 -sanity model，對 EVAL_EPISODE 中段那個 frame 跑一次 inference。
# 過關：dense reward ∈ [0.05, 0.95]、stage_probs 不是均等分布、stage_idx ∈ [1, 2]
# （中段應該大致在 stage 1 或 2，因為 ep 內 4 個 stage 線性排列）
# ============================================================
import torch, numpy as np
from lerobot.rewards.sarm.compute_rabc_weights import load_sarm_resources

with progress_stage(
    f'Inference sanity（ep {EVAL_EPISODE} 中段）',
    steps=[
        f'載入 {MODEL_REPO_ID}',
        f'抽 ep {EVAL_EPISODE} 中段 frame，preprocess + forward',
        '檢查 reward / stage_probs',
    ],
    heartbeat_seconds=30,
):
    dataset, reward_model, preprocess = load_sarm_resources(
        ANNOTATED_DATASET, MODEL_REPO_ID, device='cuda',
    )
    if hasattr(preprocess, 'eval'):
        preprocess.eval()
    for step in getattr(preprocess, 'steps', []):
        if hasattr(step, 'eval'):
            step.eval()

    cfg = reward_model.config
    image_key = cfg.image_key
    state_key = cfg.state_key
    target_idx = cfg.n_obs_steps // 2

    ep = dataset.meta.episodes[EVAL_EPISODE]
    ep_start = int(ep['dataset_from_index'])
    ep_end   = int(ep['dataset_to_index'])
    mid = (ep_start + ep_end) // 2
    sample = dataset[mid]
    task = sample.get('task', 'perform the task')

    print(f'\nep {EVAL_EPISODE}: frames {ep_start}-{ep_end} (length {ep_end-ep_start}), mid={mid}')
    print(f'task: {task!r}')

    batch = {
        image_key: sample[image_key],
        'task': task,
        'index': mid,
        'episode_index': EVAL_EPISODE,
    }
    if state_key in sample:
        batch[state_key] = sample[state_key]

    with torch.no_grad():
        processed = preprocess(batch)
        vf = processed['video_features'].to(reward_model.device)
        tf = processed['text_features'].to(reward_model.device)
        sf = processed.get('state_features')
        if sf is not None:
            sf = sf.to(reward_model.device)
        lengths = processed.get('lengths')

        reward, stages = reward_model.calculate_rewards(
            text_embeddings=tf, video_embeddings=vf,
            state_features=sf, lengths=lengths,
            return_all_frames=True, return_stages=True,
            head_mode='dense',
        )
        if isinstance(reward, torch.Tensor):
            reward = reward.cpu().numpy()
            stages = stages.cpu().numpy()
        r = float(reward[0, target_idx]) if reward.ndim == 2 else float(reward[target_idx])
        s = stages[0, target_idx, :] if stages.ndim == 3 else stages[target_idx, :]
        stage_idx = int(np.argmax(s))
        stage_max = float(s.max())

    print(f'\n=== sanity inference 結果 ===')
    print(f'  dense reward (中段 frame): {r:.4f}')
    print(f'  stage probs:              {np.round(s, 4).tolist()}')
    print(f'  argmax stage:             {stage_idx}  ({SUBTASK_NAMES[stage_idx][:40]})')
    print(f'  max stage prob:           {stage_max:.3f}')

    checks = []
    checks.append(('reward 不是 0',            abs(r) > 0.01,                 f'r={r:.4f}'))
    checks.append(('reward 不是 1',            r < 0.99,                       f'r={r:.4f}'))
    checks.append(('stage_probs 不均等',       stage_max > 0.30,               f'max={stage_max:.3f}'))
    checks.append(('中段應該 ≥ stage 1',       stage_idx >= 1,                f'argmax={stage_idx}'))

    print('\nChecks:')
    for name, ok, why in checks:
        mark = '✓' if ok else '✗'
        print(f'  {mark} {name}  ({why})')

    all_ok = all(ok for _, ok, _ in checks)
    if all_ok:
        print('\n✓✓✓ Sanity 過關 — 可以放心回 SARM_Training_Colab.ipynb 跑 5000 steps 全量訓練')
    else:
        print('\n✗ Sanity 沒過，先別跑全量。回去檢查：')
        print('  (a) Cell 5 是不是真的印 dense_subtask_names empty: 0/273')
        print('  (b) Cell 8 兩個 dense_* loss 有沒有下降')
        print('  (c) 200 步可能不夠收斂，可改 SANITY_STEPS=500 再跑一次')